<a href="https://colab.research.google.com/github/oharshg/Resume_Matching_Engine/blob/main/Resume_Matching_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Redrob AI Campus Hackathon
## Build a Resume Matching Engine

**Pipeline Overview:**
1. Normalize noisy resume skills using `SKILL_ALIASES`**bold text**
2. Deduplicate canonical skills per resume
3. Build a shared vocabulary (sorted alphabetically)
4. Compute TF-IDF vectors for resumes
5. Build binary vectors for JDs
6. Compute cosine similarity → Top 3 candidates per JD.

---
## Step 0:- Raw Data

In [ ]:
RESUMES = [
    {"id": "01", "name": "Arjun Sharma",   "raw": "Pyhton, MachineLearning, SQL, pandas, numpy, Deep-learning"},
    {"id": "02", "name": "Priya Nair",     "raw": "JavaScrpit, Reacts, Node.JS, MongoDb, REST api, HTML/CSS"},
    {"id": "03", "name": "Rahul Gupta",    "raw": "Java, Spring Boot, MySql, Microservices, Docker, kubernates"},
    {"id": "04", "name": "Sneha Patel",    "raw": "Python, TensorFlow, Keras, NLP, BERT, data-viz, matplotlib"},
    {"id": "05", "name": "Vikram Singh",   "raw": "C++, Algoritms, Data Structure, competitive programming, python"},
    {"id": "06", "name": "Ananya Krishnan","raw": "javascript, vue.js, python, flask, PostgreSQL, AWS, CI/CD"},
    {"id": "07", "name": "Karan Mehta",    "raw": "Python, Sklearn, XGboost, feature engineering, SQL, tableau"},
    {"id": "08", "name": "Deepika Rao",    "raw": "Java, Android, Kotlin, Firebase, REST, UI/UX, figma"},
    {"id": "09", "name": "Aditya Kumar",   "raw": "Reactjs, TypeScrpit, GraphQL, redux, tailwind, nodejs, jest"},
    {"id": "10", "name": "Meera Iyer",     "raw": "python, R, statistics, ML, regression, clustering, Power-BI"},
]

JDS = [
    {
        "id": "JD-1", "company": "Kakao", "role": "ML Engineer",
        "required": "Python, Machine Learning, Deep Learning, TensorFlow, PyTorch, SQL, Data Visualization",
        "preferred": "NLP, BERT, Feature Engineering, Statistics",
    },
    {
        "id": "JD-2", "company": "Naver", "role": "Backend Engineer",
        "required": "Java, Spring Boot, MySQL, PostgreSQL, Microservices, Docker, Kubernetes",
        "preferred": "REST API, CI/CD, Redis",
    },
    {
        "id": "JD-3", "company": "Line", "role": "Frontend Engineer",
        "required": "JavaScript, React, Vue, TypeScript, REST API, HTML/CSS",
        "preferred": "Node.js, GraphQL, Redux, Jest, AWS",
    },
]

print("Raw data loaded:", len(RESUMES), "resumes,", len(JDS), "JDs")

Raw data loaded: 10 resumes, 3 JDs


---
## Step 1:- SKILL_ALIASES (exact as provided)

In [ ]:
SKILL_ALIASES = {
    # Languages
    "python": "python",
    "pyhton": "python",
    "java": "java",
    "javascript": "javascript",
    "javascrpit": "javascript",
    "js": "javascript",
    "typescript": "typescript",
    "typescrpit": "typescript",
    "c++": "cpp",
    "cpp": "cpp",
    "r": "r",
    "kotlin": "kotlin",
    # ML / Data
    "machinelearning": "machine_learning",
    "machine learning": "machine_learning",
    "ml": "machine_learning",
    "sklearn": "machine_learning",
    "deeplearning": "deep_learning",
    "deep learning": "deep_learning",
    "deep-learning": "deep_learning",
    "tensorflow": "tensorflow",
    "pytorch": "pytorch",
    "keras": "keras",
    "nlp": "nlp",
    "bert": "bert",
    "xgboost": "xgboost",
    "feature engineering": "feature_engineering",
    "statistics": "statistics",
    "stats": "statistics",
    "regression": "regression",
    "clustering": "clustering",
    "data-viz": "data_visualization",
    "data visualization": "data_visualization",
    "data viz": "data_visualization",
    "matplotlib": "data_visualization",
    "tableau": "data_visualization",
    "power-bi": "data_visualization",
    "power bi": "data_visualization",
    "powerbi": "data_visualization",
    "pandas": "pandas",
    "numpy": "numpy",
    # Web — Frontend
    "react": "react",
    "reacts": "react",
    "reactjs": "react",
    "vue": "vue",
    "vue.js": "vue",
    "vuejs": "vue",
    "redux": "redux",
    "tailwind": "tailwind",
    "html/css": "html_css",
    "html css": "html_css",
    "html": "html_css",
    "css": "html_css",
    "jest": "jest",
    "graphql": "graphql",
    # Web — Backend
    "node.js": "nodejs",
    "nodejs": "nodejs",
    "node js": "nodejs",
    "flask": "flask",
    "spring boot": "spring_boot",
    "springboot": "spring_boot",
    "rest api": "rest_api",
    "rest": "rest_api",
    "restapi": "rest_api",
    "microservices": "microservices",
    # Databases
    "sql": "sql",
    "mysql": "mysql",
    "mysq": "mysql",
    "postgresql": "postgresql",
    "postgres": "postgresql",
    "mongodb": "mongodb",
    "redis": "redis",
    # DevOps / Cloud
    "docker": "docker",
    "kubernetes": "kubernetes",
    "kubernates": "kubernetes",
    "k8s": "kubernetes",
    "ci/cd": "ci_cd",
    "cicd": "ci_cd",
    "ci cd": "ci_cd",
    "aws": "aws",
    # Mobile
    "android": "android",
    "firebase": "firebase",
    # CS Fundamentals
    "algorithms": "algorithms",
    "algoritms": "algorithms",
    "data structure": "data_structures",
    "data structures": "data_structures",
    "competitive programming": "competitive_programming",
    # Design
    "ui/ux": "ui_ux",
    "ui ux": "ui_ux",
    "figma": "figma",
}

# Sort multi-word phrases longest-first so they're matched before single tokens
SORTED_ALIASES = sorted(SKILL_ALIASES.keys(), key=lambda k: (-len(k.split()), k))

print(f" SKILL_ALIASES loaded: {len(SKILL_ALIASES)} entries")
print(f" Multi-word phrases matched first (longest → shortest)")

 SKILL_ALIASES loaded: 89 entries
 Multi-word phrases matched first (longest → shortest)


---
## Step 2:- Normalize & Deduplicate Skills

In [ ]:
import re

def normalize_skills(raw_string):
    """
    1. Split on commas → individual tokens
    2. Lowercase + strip each token
    3. Try multi-word alias matches first (longest-first order)
    4. Fall back to single-token alias lookup
    5. Discard tokens not in SKILL_ALIASES
    6. Deduplicate (preserve first-seen order)
    """
    tokens = [t.strip().lower() for t in raw_string.split(",")]

    canonical = []
    seen = set()

    for token in tokens:
        matched = False
        # Try every alias key (sorted longest-phrase-first)
        for alias_key in SORTED_ALIASES:
            if alias_key == token or token == alias_key:
                skill = SKILL_ALIASES[alias_key]
                if skill not in seen:
                    canonical.append(skill)
                    seen.add(skill)
                matched = True
                break
        # Token not in alias map → discard (no else branch needed)

    return canonical


# Apply normalization to all resumes
for r in RESUMES:
    r["skills"] = normalize_skills(r["raw"])

# Display results
print("=" * 65)
print(f"{'ID':<4} {'Candidate':<18} Normalized Skills")
print("=" * 65)
for r in RESUMES:
    print(f"{r['id']:<4} {r['name']:<18} {r['skills']}")
print("=" * 65)

ID   Candidate          Normalized Skills
01   Arjun Sharma       ['python', 'machine_learning', 'sql', 'pandas', 'numpy', 'deep_learning']
02   Priya Nair         ['javascript', 'react', 'nodejs', 'mongodb', 'rest_api', 'html_css']
03   Rahul Gupta        ['java', 'spring_boot', 'mysql', 'microservices', 'docker', 'kubernetes']
04   Sneha Patel        ['python', 'tensorflow', 'keras', 'nlp', 'bert', 'data_visualization']
05   Vikram Singh       ['cpp', 'algorithms', 'data_structures', 'competitive_programming', 'python']
06   Ananya Krishnan    ['javascript', 'vue', 'python', 'flask', 'postgresql', 'aws', 'ci_cd']
07   Karan Mehta        ['python', 'machine_learning', 'xgboost', 'feature_engineering', 'sql', 'data_visualization']
08   Deepika Rao        ['java', 'android', 'kotlin', 'firebase', 'rest_api', 'ui_ux', 'figma']
09   Aditya Kumar       ['react', 'typescript', 'graphql', 'redux', 'tailwind', 'nodejs', 'jest']
10   Meera Iyer         ['python', 'r', 'statistics', 'machine_le

---
## Step 3:- Build Shared Vocabulary

In [ ]:
# Collect all unique skills across all resumes → sort alphabetically
all_skills = set()
for r in RESUMES:
    all_skills.update(r["skills"])

VOCAB = sorted(all_skills)          # alphabetical order → consistent indexing
VOCAB_INDEX = {s: i for i, s in enumerate(VOCAB)}

print(f"Vocabulary size: {len(VOCAB)} unique canonical skills\n")
for i, skill in enumerate(VOCAB):
    print(f"  [{i:02d}] {skill}")

Vocabulary size: 48 unique canonical skills

  [00] algorithms
  [01] android
  [02] aws
  [03] bert
  [04] ci_cd
  [05] clustering
  [06] competitive_programming
  [07] cpp
  [08] data_structures
  [09] data_visualization
  [10] deep_learning
  [11] docker
  [12] feature_engineering
  [13] figma
  [14] firebase
  [15] flask
  [16] graphql
  [17] html_css
  [18] java
  [19] javascript
  [20] jest
  [21] keras
  [22] kotlin
  [23] kubernetes
  [24] machine_learning
  [25] microservices
  [26] mongodb
  [27] mysql
  [28] nlp
  [29] nodejs
  [30] numpy
  [31] pandas
  [32] postgresql
  [33] python
  [34] r
  [35] react
  [36] redux
  [37] regression
  [38] rest_api
  [39] spring_boot
  [40] sql
  [41] statistics
  [42] tailwind
  [43] tensorflow
  [44] typescript
  [45] ui_ux
  [46] vue
  [47] xgboost


---
## Step 4:- Compute TF-IDF Vectors for Resumes

```
TF(skill, resume) = 1 / N          # N = total unique skills in that resume
IDF(skill)        = ln(10 / df)    # df = number of resumes containing skill
TF-IDF            = TF × IDF
```

In [ ]:
import math

N_DOCS = len(RESUMES)   # 10

# ── Document Frequency (df) per skill ────────────────────────────────────────
df = {skill: 0 for skill in VOCAB}
for r in RESUMES:
    for skill in r["skills"]:
        df[skill] += 1

# ── IDF ───────────────────────────────────────────────────────────────────────
idf = {skill: math.log(N_DOCS / df[skill]) for skill in VOCAB}

print("IDF values per skill:")
print("-" * 42)
for skill in VOCAB:
    print(f"  {skill:<25}  df={df[skill]}  IDF={idf[skill]:.6f}")

IDF values per skill:
------------------------------------------
  algorithms                 df=1  IDF=2.302585
  android                    df=1  IDF=2.302585
  aws                        df=1  IDF=2.302585
  bert                       df=1  IDF=2.302585
  ci_cd                      df=1  IDF=2.302585
  clustering                 df=1  IDF=2.302585
  competitive_programming    df=1  IDF=2.302585
  cpp                        df=1  IDF=2.302585
  data_structures            df=1  IDF=2.302585
  data_visualization         df=3  IDF=1.203973
  deep_learning              df=1  IDF=2.302585
  docker                     df=1  IDF=2.302585
  feature_engineering        df=1  IDF=2.302585
  figma                      df=1  IDF=2.302585
  firebase                   df=1  IDF=2.302585
  flask                      df=1  IDF=2.302585
  graphql                    df=1  IDF=2.302585
  html_css                   df=1  IDF=2.302585
  java                       df=2  IDF=1.609438
  javascript           

In [ ]:
# ── TF-IDF vectors ────────────────────────────────────────────────────────────
def build_tfidf_vector(resume):
    skills = resume["skills"]
    N = len(skills)                    # unique skill count after deduplication
    vec = [0.0] * len(VOCAB)
    for skill in skills:
        idx = VOCAB_INDEX[skill]
        tf = 1.0 / N
        vec[idx] = tf * idf[skill]
    return vec

for r in RESUMES:
    r["tfidf"] = build_tfidf_vector(r)

# ── Preview TF-IDF (non-zero entries only) ────────────────────────────────────
print("TF-IDF vectors (non-zero entries):")
print("=" * 65)
for r in RESUMES:
    N = len(r["skills"])
    print(f"\n{r['name']} (N={N}):")
    for skill in r["skills"]:
        idx = VOCAB_INDEX[skill]
        val = r["tfidf"][idx]
        print(f"   {skill:<25} TF={1/N:.4f}  IDF={idf[skill]:.4f}  TF-IDF={val:.6f}")

TF-IDF vectors (non-zero entries):

Arjun Sharma (N=6):
   python                    TF=0.1667  IDF=0.5108  TF-IDF=0.085138
   machine_learning          TF=0.1667  IDF=1.2040  TF-IDF=0.200662
   sql                       TF=0.1667  IDF=1.6094  TF-IDF=0.268240
   pandas                    TF=0.1667  IDF=2.3026  TF-IDF=0.383764
   numpy                     TF=0.1667  IDF=2.3026  TF-IDF=0.383764
   deep_learning             TF=0.1667  IDF=2.3026  TF-IDF=0.383764

Priya Nair (N=6):
   javascript                TF=0.1667  IDF=1.6094  TF-IDF=0.268240
   react                     TF=0.1667  IDF=1.6094  TF-IDF=0.268240
   nodejs                    TF=0.1667  IDF=1.6094  TF-IDF=0.268240
   mongodb                   TF=0.1667  IDF=2.3026  TF-IDF=0.383764
   rest_api                  TF=0.1667  IDF=1.6094  TF-IDF=0.268240
   html_css                  TF=0.1667  IDF=2.3026  TF-IDF=0.383764

Rahul Gupta (N=6):
   java                      TF=0.1667  IDF=1.6094  TF-IDF=0.268240
   spring_boot       

---
## Step 5:- Build Binary JD Vectors

In [ ]:
def normalize_jd_skills(required_str, preferred_str):
    """Normalize JD skill strings using the same alias map."""
    combined = required_str + ", " + preferred_str
    return normalize_skills(combined)


def build_jd_binary_vector(jd):
    """
    Binary vector over VOCAB:
      1 if the canonical JD skill is in VOCAB, else 0
    (Skills that didn't appear in any resume are not in VOCAB → ignored)
    """
    jd_skills = normalize_jd_skills(jd["required"], jd["preferred"])
    vec = [0] * len(VOCAB)
    matched = []
    for skill in jd_skills:
        if skill in VOCAB_INDEX:
            vec[VOCAB_INDEX[skill]] = 1
            matched.append(skill)
    return vec, matched


for jd in JDS:
    jd["vec"], jd["matched_skills"] = build_jd_binary_vector(jd)

print("JD Binary Vectors (skills present in vocabulary):")
print("=" * 65)
for jd in JDS:
    print(f"\n{jd['id']} — {jd['company']} ({jd['role']}):")
    print(f"  Matched vocab skills: {jd['matched_skills']}")

JD Binary Vectors (skills present in vocabulary):

JD-1 — Kakao (ML Engineer):
  Matched vocab skills: ['python', 'machine_learning', 'deep_learning', 'tensorflow', 'sql', 'data_visualization', 'nlp', 'bert', 'feature_engineering', 'statistics']

JD-2 — Naver (Backend Engineer):
  Matched vocab skills: ['java', 'spring_boot', 'mysql', 'postgresql', 'microservices', 'docker', 'kubernetes', 'rest_api', 'ci_cd']

JD-3 — Line (Frontend Engineer):
  Matched vocab skills: ['javascript', 'react', 'vue', 'typescript', 'rest_api', 'html_css', 'nodejs', 'graphql', 'redux', 'jest', 'aws']


---
## Step 6:- Cosine Similarity & Ranking

```
Cosine(A, B) = (A · B) / (|A| × |B|)
  A = resume TF-IDF vector
  B = JD binary vector
```

In [ ]:
def dot_product(a, b):
    return sum(x * y for x, y in zip(a, b))

def euclidean_norm(v):
    return math.sqrt(sum(x * x for x in v))

def cosine_similarity(a, b):
    norm_a = euclidean_norm(a)
    norm_b = euclidean_norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot_product(a, b) / (norm_a * norm_b)


# ── Compute similarity scores ─────────────────────────────────────────────────
results = {}

for jd in JDS:
    scores = []
    for r in RESUMES:
        sim = cosine_similarity(r["tfidf"], jd["vec"])
        scores.append((r["name"], sim))

    # Sort: descending by score, alphabetically by name for ties
    scores.sort(key=lambda x: (-round(x[1], 10), x[0]))
    results[jd["id"]] = scores

# ── Detailed scores table ─────────────────────────────────────────────────────
print("Cosine Similarity — All Candidates:\n")
for jd in JDS:
    print(f"{jd['id']} — {jd['company']} ({jd['role']}):")
    print(f"  {'Rank':<6} {'Candidate':<20} {'Score':>8}")
    print(f"  {'----':<6} {'---------':<20} {'-----':>8}")
    for rank, (name, score) in enumerate(results[jd["id"]], 1):
        marker = " TOP 3" if rank <= 3 else ""
        print(f"  {rank:<6} {name:<20} {score:>8.4f}{marker}")
    print()

Cosine Similarity — All Candidates:

JD-1 — Kakao (ML Engineer):
  Rank   Candidate               Score
  ----   ---------               -----
  1      Sneha Patel            0.5696 TOP 3
  2      Karan Mehta            0.5341 TOP 3
  3      Arjun Sharma           0.3958 TOP 3
  4      Meera Iyer             0.3345
  5      Vikram Singh           0.0349
  6      Ananya Krishnan        0.0298
  7      Aditya Kumar           0.0000
  8      Deepika Rao            0.0000
  9      Priya Nair             0.0000
  10     Rahul Gupta            0.0000

JD-2 — Naver (Backend Engineer):
  Rank   Candidate               Score
  ----   ---------               -----
  1      Rahul Gupta            0.8109 TOP 3
  2      Ananya Krishnan        0.2833 TOP 3
  3      Deepika Rao            0.1906 TOP 3
  4      Priya Nair             0.1172
  5      Aditya Kumar           0.0000
  6      Arjun Sharma           0.0000
  7      Karan Mehta            0.0000
  8      Meera Iyer             0.0000
  9    

##
Final Output

In [ ]:
print("=" * 55)
print("         FINAL RESULTS — TOP 3 CANDIDATES PER JD")
print("=" * 55)

for jd in JDS:
    top3 = results[jd["id"]][:3]
    label = f"{jd['id']} — {jd['company']} ({jd['role']})"
    ranking = ", ".join(f"{name}({score:.2f})" for name, score in top3)
    print(f"\n{label}")
    print(ranking)

print("\n" + "=" * 55)

         FINAL RESULTS — TOP 3 CANDIDATES PER JD

JD-1 — Kakao (ML Engineer)
Sneha Patel(0.57), Karan Mehta(0.53), Arjun Sharma(0.40)

JD-2 — Naver (Backend Engineer)
Rahul Gupta(0.81), Ananya Krishnan(0.28), Deepika Rao(0.19)

JD-3 — Line (Frontend Engineer)
Aditya Kumar(0.67), Priya Nair(0.58), Ananya Krishnan(0.35)



---
## Debug — Vocabulary & df Summary

In [ ]:
print(f"{'Skill':<28} {'df':>4}  {'IDF':>10}")
print("-" * 46)
for skill in VOCAB:
    print(f"{skill:<28} {df[skill]:>4}  {idf[skill]:>10.6f}")

Skill                          df         IDF
----------------------------------------------
algorithms                      1    2.302585
android                         1    2.302585
aws                             1    2.302585
bert                            1    2.302585
ci_cd                           1    2.302585
clustering                      1    2.302585
competitive_programming         1    2.302585
cpp                             1    2.302585
data_structures                 1    2.302585
data_visualization              3    1.203973
deep_learning                   1    2.302585
docker                          1    2.302585
feature_engineering             1    2.302585
figma                           1    2.302585
firebase                        1    2.302585
flask                           1    2.302585
graphql                         1    2.302585
html_css                        1    2.302585
java                            2    1.609438
javascript                      2